# Data Scientist

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import pandas as pd

df = pd.read_csv('../data/air_fryers_clean_brand_year.csv')

y = df['log_brand_share']

chars = ['compact_share', 'dual_basket_share', 'oven_style_share',
         'rotisserie_share', 'window_share']
 
X = pd.concat([
    df[['avg_price', 'avg_rating'] + chars],
    pd.get_dummies(df['brand'], prefix='b', drop_first=True),
    pd.get_dummies(df['year'],  prefix='y', drop_first=True),
], axis=1)

model = LinearRegression()
model.fit(X, y)

table = pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_})
print(table)
print(f"\nR² = {model.score(X, y):.4f}")


              feature  coefficient
0           avg_price    -0.037668
1          avg_rating     0.287517
2       compact_share     9.815304
3   dual_basket_share    -9.509686
4    oven_style_share     1.941774
5    rotisserie_share    -5.674054
6        window_share    12.880298
7            b_cosori     2.551946
8         b_cuisinart     6.422436
9              b_dash     0.176655
10       b_gowise usa     3.938996
11      b_instant_pot     4.626260
12            b_ninja     5.838705
13           b_nuwave     3.544883
14            b_oster     3.928074
15          b_ultrean     0.942399
16             y_2020     0.119071
17             y_2021     0.041900
18             y_2022    -0.098860
19             y_2023    -0.003307

R² = 0.7635



Questions:

***1. What is the estimated price coefficient, $\hat{\beta}_{price}$?***

 $\hat{\beta}_{price}$ = −0.037668. This means that a $1 increase in average price is associated with roughly a 3.8% decrease in brand share, holding brand, year, rating, and product features.

***2. Is it negative? Why is that important?***
Yes, it's negative. Demand should slope downward, so a higher price should reduce share. This also acts as a sanity check against the model, making sure that we comply with the law of demand.

***3. Which product features are associated with higher demand?***
A greater coefficient indicates that there is a higher demand. The product features with positive coefficients are window_share (+12.88), compact_share (+9.82), oven_style_share (+1.94), and avg_rating (+0.29). window_share has the largest effect, which tells us that consumers strongly prefer air fryers with viewing windows. Two characteristics are associated with lower demand are dual_basket_share (−9.51) and rotisserie_share (−5.67). 

***4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.***
Cuisinart (+6.42), Ninja (+5.84), and Instant Pot (+4.63) have the largest dummy coefficients. However, all 9 brand dummies are positive, but Cuisinart, Ninja, and Instant Pot have the largest coefficients. Because these are interpreted as relative to the dropped brand (chefman), this means that Cuisinart's log share of the holding price, rating, features, and year fixed is about 6.4 higher than chefman's. 

***5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.***
Relative to the dropped year (2019), the year coefficients are 2020 (+0.119), 2021 (+0.042), 2023 (−0.003), and 2022 (−0.099). 2020 is the largest, which tells us that baseline air fryer demand was the highest that year. The year effects are significantly less than brand effects, so brand matters far more than how old an air fryer is. 

***6. What is the model's $R^2$?***

$R^2=0.7635$. The model explains about 76% of the variation in log brand share across the 50 brand-year cells. The remaining 24% reflects within-brand, within-year variation that price, rating, and the five characteristic shares don't capture.


This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.